## Overview: Lab 8 - Extending Local Search Techniques

> In the previous lab, you were introduced to the fundamental concepts of **Local Search Algorithms**, specifically focusing on **Steepest Ascent Hill Climbing**. You learned how these algorithms differ from traditional search strategies by exploring the state space locally, without maintaining a full search tree or queue.
>
> This lab builds upon that foundation by introducing several advanced local search techniques. We will explore how these methods enhance the basic hill-climbing approach, offering improved capabilities to navigate complex search spaces and avoid local optima.
>
> We will continue to apply these strategies to two classic AI problems:
>
> -   **The Traveling Salesperson Problem (TSP)**: Finding the shortest possible route that visits each city exactly once and returns to the starting city in a complete graph.
> -   **The 8-Queens Problem**: Placing eight queens on a chessboard such that no two queens threaten each other.

## Objectives

> -   **Extend Local Search Implementations**: Add to the existing steepest ascent implementation by incorporating stochastic hill climbing, first choice hill climbing, random restart hill climbing, and simulated_annealing.
> -   **Implement and Evaluate Advanced Local Search Strategies**:
>     -   **Stochastic Hill Climbing (HC)**: Randomly select among neighbors with better evaluations.
>     -   **First Choice Hill Climbing**: Select the first neighbor that improves the current state.
>     -   **Random Restart Hill Climbing**: Perform multiple hill-climbing searches from random initial states.
>     -   **Simulated annealing**: probabilistically accept worse moves to escape local optima.
> -   **Analyze and Compare Algorithm Performance**: Evaluate the effectiveness of each strategy on TSP and the 8-Queens Problem.
> -   **Understand the Trade-offs**: Discuss the strengths and weaknesses of each local search method, including their ability to escape local optima and the computational costs involved.

## 🛠️ Your Tasks:

### ✅ 1. Complete the Following Methods:

> -   Ensure the Traveling Salesperson Problem (TSP) class and Eight Queens Problem class from the previous lab are correctly implemented.
> -   Enhance the \`_hill_climbing\` method to include steepest, stochastic, and first choice hill climbing.
> -   Implement the \`random_restart_hill_climbing\` method.
> -   Implement the \`_simulated_annealing\` method.
> -   Implement the \`local_search\` method to dispatch the correct local search strategy.

### ✅ 2. Run the Provided Tests:

> Test your implementation on both:
> -   \`tsp_problem\` (for TSP)
> -   \`queens_problem\` (for 8-Queens)
>
> Ensure your tests cover all implemented strategies (Steepest, Stochastic, First Choice, Random Restart, Simulated Annealing).

### ✅ 3. Analyze the Results:

> -   Compare the outputs of each strategy to understand their performance characteristics and how they differ from the steepest ascent method you implemented in the previous lab.
> -   Analyze how each strategy explores the state space and whether it converges to a valid solution.
> -   Discuss the impact of parameters like \`initial_temperature\`, \`cooling_rate\`, \`max_iterations\`, and the number of restarts on the quality of the solution.
> -   Reflect on the trade-offs between exploration and exploitation in each strategy.
> -   Discuss the problems each strategy is best suited for, and how they improve upon the basic steepest descent method.

In [15]:
import random
import math
import copy
import time  # Added for potential timing/logging if needed

# ------------------------------------------------------------------------------
# Candidate Class: Represents a potential solution state and its evaluation
# ------------------------------------------------------------------------------
class Candidate:
    """
    Represents a candidate solution in a local search context.
    """
    def __init__(self, state, value):
        self.state = state
        self.value = value

    def __repr__(self):
        state_repr = str(self.state)
        if len(state_repr) > 50:
            state_repr = state_repr[:47] + "..."
        value_repr = f"{self.value:.4f}" if isinstance(self.value, float) else str(self.value)
        return f"Candidate(state={state_repr}, value={value_repr})"

# ------------------------------------------------------------------------------
# Traveling Salesperson Problem (TSP) Implementation
# ------------------------------------------------------------------------------
class TSPProblem:
    def __init__(self, cities):
        if not cities:
            raise ValueError("City dictionary cannot be empty.")
        self.cities = cities
        self.city_list = list(cities.keys())
        # Generate initial state *once* upon creation
        self.initial_state = self._generate_random_state()

    def _generate_random_state(self):
        random_tour = self.city_list[:]
        random.shuffle(random_tour)
        return random_tour

    def evaluate(self, state):
        total_distance = 0.0
        n = len(state)
        if n < 2:
            return 0.0
        for i in range(n):
            city1_name = state[i]
            city2_name = state[(i + 1) % n]  # Handle wrap-around
            coord1 = self.cities[city1_name]
            coord2 = self.cities[city2_name]
            total_distance += self._distance(coord1, coord2)
        return total_distance

    def _distance(self, coord1, coord2):
        return math.sqrt((coord1[0] - coord2[0]) ** 2 + (coord1[1] - coord2[1]) ** 2)

    def generate_neighbors(self, state):
        neighbors = []
        n = len(state)
        if n < 2:
            return []
        for i in range(n - 1):
            for j in range(i + 1, n):
                neighbor = state[:]  # Shallow copy
                neighbor[i], neighbor[j] = neighbor[j], neighbor[i]  # Swap two cities
                neighbors.append(neighbor)
        return neighbors

    def generate_random_neighbor(self, state):
        """
        Generates a single random neighbor of the current TSP tour.

        Input:
          - state: A list representing a tour where each element is a city name.
                   Example: ['A', 'B', 'C', 'D']

        Output:
          - A new list that is a slight variation of the input state. Specifically,
            it should be identical to the input except that two cities have been swapped.
          - Example: Given state ['A', 'B', 'C', 'D'], one valid output is ['A', 'C', 'B', 'D'].

        """
        n = len(state)
        if n < 2:
            return state[:]  # Return a copy if only one or no cities

        # BEGIN: Student implementation required from here.
        # 1. Pick two distinct indices, i and j.
        # 2. Create a copy of the state.
        # 3. Swap the cities at indices i and j.
        # 4. Return the new neighbor state.
        i,j = random.sample(range(0,n),2)
        state_copy = state.copy()
        state_copy[i], state_copy[j]= state_copy[j],state_copy[i]
        return state_copy

# ------------------------------------------------------------------------------
# Eight Queens Problem Implementation
# ------------------------------------------------------------------------------
class EightQueensProblem:
    def __init__(self, size=8, initial_state=None):
        self.size = size
        if initial_state:
            if len(initial_state) != self.size or len(set(initial_state)) != self.size:
                raise ValueError(f"Initial state must be a valid permutation of 0..{size-1}")
            self.initial_state = list(initial_state)  # Ensure it's a list
        else:
            # Generate initial state *once* upon creation
            self.initial_state = self._generate_random_state()

    def _generate_random_state(self):
        state = list(range(self.size))
        random.shuffle(state)
        return state

    def evaluate(self, state):
        """
        Calculates the number of pairs of attacking queens.

        Input:
          - state: A list representing queen positions. Each index represents a row and the value at that index is the column.
                   Example for an 8-queens problem: [4, 2, 7, 3, 6, 8, 5, 1]

        Output:
          - An integer count of diagonal conflicts (lower is better; 0 means a correct solution).

        The function checks every pair of queens for conflicts.
        """
        conflicts = 0
        n = self.size
        for i in range(n):
            for j in range(i + 1, n):
                if abs(state[i] - state[j]) == abs(i - j):
                    conflicts += 1
        return conflicts  # Goal is to have 0 conflicts

    def generate_neighbors(self, state):
        neighbors = []
        n = self.size
        if n < 2:
            return []
        for i in range(n - 1):
            for j in range(i + 1, n):
                neighbor = state[:]  # Shallow copy
                neighbor[i], neighbor[j] = neighbor[j], neighbor[i]  # Swap queens in columns
                neighbors.append(neighbor)
        return neighbors

    def generate_random_neighbor(self, state):
        """
        Generates a single random neighbor of the current Eight Queens configuration.

        Input:
          - state: A list representing the board configuration. Each element is the column where the queen is placed in that row.
                   Example: [0, 4, 7, 5, 2, 6, 1, 3]

        Output:
          - A new list where exactly two positions are swapped.
          - Example: Given [0, 4, 7, 5, 2, 6, 1, 3], one valid output is [0, 7, 4, 5, 2, 6, 1, 3].
        """
        n = self.size
        if n < 2:
            return state[:]  # Return a copy

        # BEGIN: Student implementation required from here.
        # 1. Select two unique random indices.
        # 2. Create a copy of the state.
        # 3. Swap the values at the indices.
        # 4. Return the newly generated neighbor.
        i,j = random.sample(range(0, n-1), 2)
        copy_state = state.copy()
        copy_state[i],copy_state[j]=copy_state[j],copy_state[i]
        return copy_state
        

# ------------------------------------------------------------------------------
# Local Search Algorithms
# ------------------------------------------------------------------------------
def _hill_climbing(problem, selection_strategy="steepest"):
    """
    Performs hill climbing local search based on the given selection strategy.

    Input:
      - problem: An instance of a problem class that provides:
          * initial_state: the starting state
          * evaluate(state): a function to compute the state’s score
          * generate_neighbors(state): a function that returns a list of neighboring states
      - selection_strategy: A string indicating the strategy to use. Valid options are:
          "steepest"   - choose the neighbor with the best (lowest) evaluation.
          "stochastic" - randomly choose among the neighbors that improve the solution.
          "first_choice" - take the first neighbor found that improves the evaluation.

    Output:
      - Returns a Candidate (object with state and its evaluation value) representing a local optimum found.
    """
    # 1. Set current = initial state with its evaluation
    current_state = problem.initial_state[:]  # Work with a copy
    current_value = problem.evaluate(current_state)
    current_candidate = Candidate(current_state, current_value)

    # 2. Repeat:
    #     - Generate all neighbors of current_candidate
    #     - Go through each neighbor, evaluate it, and create a list of Candidate objects.

    #     - Choose next based on selection_strategy:
    #         - "steepest": pick best neighbor if better than current
    #         - "stochastic": randomly pick one that's better than current
    #         - "first_choice": go through neighbors and pick first better one

    #     - If no better neighbor is found:
    #         stop and return current

    #     - Otherwise:
    #         update current to the chosen neighbor

    # 3. Return current as the best found
    neighbors = problem.generate_neighbors(current_candidate.state)
    candidates = [Candidate(neighbor, problem.evaluate(neighbor)) for neighbor in neighbors]
    # Filter candidates to only those that improve the solution
    improving_candidates = [c for c in candidates if c.value < current_candidate.value]
    if not improving_candidates:
        return current_candidate  # No better neighbor found, return current
    if selection_strategy == "steepest":
        # Pick the best neighbor (steepest ascent)
        next_candidate = min(improving_candidates, key=lambda c: c.value)
    elif selection_strategy == "stochastic":
        # Randomly select one of the improving candidates
        next_candidate = random.choice(improving_candidates)
    elif selection_strategy == "first_choice":
        # Pick the first improving candidate
        next_candidate = improving_candidates[0]
    elif selection_strategy == "random_restart":
        # Randomly select one of the neighbors (not necessarily improving)
        next_candidate = random.choice(candidates)

    else:
        raise ValueError(f"Unknown selection strategy: {selection_strategy}")
    # Update current_candidate to the next candidate  

    return next_candidate  # Return the best candidate found

def random_restart_hill_climbing(problem_instance, num_restarts=100, base_strategy="steepest"):
    """
    Performs hill climbing multiple times with random restarts to overcome local optima.

    Input:
      - problem_instance: an instance of a problem (TSPProblem or EightQueensProblem) that will be used to generate new initial states.
      - num_restarts: an integer, the number of random restarts to perform.
      - base_strategy: a string indicating the hill climbing strategy ("steepest", "stochastic", "first_choice") to use for each run.

    Output:
      - Returns the best Candidate (with the lowest evaluation value) found across all restarts.
    """
    best_candidate_overall = None  # This will hold the best solution found across all restarts
    problem_class = type(problem_instance)  # Get the class/type of the problem instance

    # Repeat for the specified number of restarts
    for i in range(num_restarts):
        # Create a new problem instance with a new random initial state
        try:
            # Step 1: Check the type of the problem (TSP or EightQueens)
            if isinstance(problem_instance, TSPProblem):
                # If TSP problem, create a new instance with random initial state
                current_problem = TSPProblem(problem_instance.cities)
            elif isinstance(problem_instance, EightQueensProblem):
                # If EightQueensProblem, create a new instance with random initial state
                current_problem = EightQueensProblem(problem_instance.size)
            else:
                # Handle unknown problem types
                print("Error: Unknown problem type for random restart !!")
        except Exception as e:
            # Handle any errors while creating new instances
            print(f"Error creating new problem instance: {e}.")


        # Step 2: Perform hill climbing on the new problem instance using
        # the _hill_climbing function with the given selection_strategy
        # to find the best candidate solution for this restart.
        candidate_this_restart = _hill_climbing(current_problem, selection_strategy=base_strategy)

        # Step 3: Update the best candidate if this restart resulted in a better solution
        if best_candidate_overall is None or candidate_this_restart.value < best_candidate_overall.value:
            best_candidate_overall = candidate_this_restart
    # Step 4: After all restarts, return the best candidate found
    return best_candidate_overall


def _simulated_annealing(problem, initial_temperature, cooling_rate, max_iterations):
    """
    Performs the simulated annealing search.

    Input:
      - problem: a problem instance which provides:
             * initial_state (a starting solution)
             * evaluate(state): a function that returns the evaluation (cost) of the state.
             * generate_random_neighbor(state): returns a randomly generated neighbor state.
      - initial_temperature: a float representing the starting temperature for the annealing schedule.
      - cooling_rate: a float between 0 and 1 that reduces the temperature at each iteration (e.g., 0.999).
      - max_iterations: an integer representing the maximum iterations to execute.

    Output:
      - Returns a Candidate representing the best solution found (state and its evaluation value).
    """
    # Step 1: Initialize the current state with problem.initial_state and compute its evaluation.
    current_state = problem.initial_state[:]  # Work with a copy
    current_value = problem.evaluate(current_state)
    current_candidate = Candidate(current_state, current_value)

    # Step 2: Set best_candidate equal to the current state as the starting best solution and temperature equal initial_temperature
    best_candidate = current_candidate  # Track the best solution overall

    temperature = initial_temperature


    # Step 3: Repeat until max_iterations is reached:
    #    a. If the temperature falls below a small threshold (e.g., 1e-6), stop the process (break).
    #    b. Generate one random neighbor using problem.generate_random_neighbor.
    #    c. Evaluate the generated neighbor's state.
    #    d. Calculate the change in evaluation: delta_e = neighbor_value - current_value.
    #    e. If the neighbor is better (delta_e < 0), move to the neighbor and update best_candidate if needed.
    #    f. If the neighbor is worse, accept it with a probability exp(-delta_e / temperature) (using math.exp).
    #    g. Reduce the temperature by multiplying it by cooling_rate for the next iteration.
    for iteration in range(max_iterations):
        if temperature < 1e-6:
            break
        neighbor_state = problem.generate_random_neighbor(current_candidate.state)
        neighbor_value = problem.evaluate(neighbor_state)
        delta_e = neighbor_value - current_candidate.value
        if delta_e < 0:
            # Move to the neighbor if it's better
            current_candidate = Candidate(neighbor_state, neighbor_value)
            if current_candidate.value < best_candidate.value:
                best_candidate = current_candidate
        else:
            # Accept the worse neighbor with a probability
            acceptance_probability = math.exp(-delta_e / temperature)
            if random.random() < acceptance_probability:
                current_candidate = Candidate(neighbor_state, neighbor_value)
        # Reduce the temperature
        temperature *= cooling_rate

    # Step 4: After the loop completes, return the best_candidate found during the process.
    return best_candidate

def local_search(problem,
                 strategy="steepest",
                 num_restarts=10, base_restart_strategy="steepest",
                 initial_temperature_sa=50, cooling_rate_sa=0.9990, max_iterations_sa=5000):
    """
    Main dispatcher for local search algorithms.

    Input:
      - problem: an instance of TSPProblem or EightQueensProblem.
      - strategy: a string determining which algorithm to use ("steepest", "stochastic", "first_choice",
                  "random_restart", or "simulated_annealing").
      - num_restarts: integer, only relevant when strategy is "random_restart".
      - base_restart_strategy: hill climbing strategy to use for each random restart.
      - initial_temperature_sa, cooling_rate_sa, max_iterations_sa: parameters for simulated annealing.

    Output:
      - Returns a Candidate object representing the best solution found.
    """
    if strategy in ["steepest", "stochastic", "first_choice"]:
        print(f"\n=== Running: {strategy.capitalize()} Hill Climbing ===")
        return _hill_climbing(problem, selection_strategy=strategy)
    elif strategy == "random_restart":
        print(f"\n=== Running: Random Restart Hill Climbing ({num_restarts} restarts) ===")
        return random_restart_hill_climbing(problem, num_restarts, base_restart_strategy)
    elif strategy == "simulated_annealing":
        print("\n=== Running: Simulated Annealing ===")
        return _simulated_annealing(problem, initial_temperature_sa, cooling_rate_sa, max_iterations_sa)
    else:
        raise ValueError(f"Unknown strategy: {strategy}. Choose 'steepest', 'stochastic', "
                         f"'first_choice', 'random_restart', or 'simulated_annealing'.")

# ------------------------------------------------------------------------------
# Test Functions
# ------------------------------------------------------------------------------
def test_tsp():
    print("\n----- TSP Local Search Test -----")

    cities = {
        'A': (0, 0), 'B': (2, 10), 'C': (5, 4), 'D': (6, 15),
        'E': (9, 3), 'F': (14, 10), 'G': (15, 0), 'H': (20, 5),
        'I': (22, 12), 'J': (25, 2), 'K': (28, 18), 'L': (30, 8),
        'M': (33, 15), 'N': (35, 3), 'O': (38, 10), 'P': (40, 0),
        'Q': (42, 7), 'R': (45, 14), 'S': (48, 5), 'T': (50, 16)
    }
    num_restarts_tsp = iterations_tsp = 50

    tsp_problem_base = TSPProblem(cities)  # Create one instance to show initial state
    print(f"Sample Initial TSP tour: {tsp_problem_base.initial_state}")
    print(f"Sample Initial tour cost: {tsp_problem_base.evaluate(tsp_problem_base.initial_state):.2f}")

    # --- Test Hill Climbing Variants ---
    candidate_steepest = local_search(TSPProblem(cities), strategy="steepest")
    print(f"Steepest Ascent Final tour cost: {candidate_steepest.value:.2f}")

    candidate_stochastic = local_search(TSPProblem(cities), strategy="stochastic")
    print(f"Stochastic HC Final tour cost: {candidate_stochastic.value:.2f}")

    candidate_first_choice = local_search(TSPProblem(cities), strategy="first_choice")
    print(f"First Choice HC Final tour cost: {candidate_first_choice.value:.2f}")

    # --- Test Random Restart ---
    candidate_restart_steepest = local_search(TSPProblem(cities), strategy="random_restart", num_restarts=num_restarts_tsp, base_restart_strategy="steepest")
    print(f"Random Restart (base: Steepest) Final tour cost: {candidate_restart_steepest.value:.2f}")

    candidate_restart_stochastic = local_search(TSPProblem(cities), strategy="random_restart", num_restarts=num_restarts_tsp, base_restart_strategy="stochastic")
    print(f"Random Restart (base: Stochastic) Final tour cost: {candidate_restart_stochastic.value:.2f}")

    candidate_restart_first_choice = local_search(TSPProblem(cities), strategy="random_restart", num_restarts=num_restarts_tsp, base_restart_strategy="first_choice")
    print(f"Random Restart (base: First Choice) Final tour cost: {candidate_restart_first_choice.value:.2f}")

    candidate_sa = local_search(TSPProblem(cities), strategy="simulated_annealing")
    print(f"Simulated Annealing Final tour cost: {candidate_sa.value:.2f}")
    print(f"(Best SA state found: {candidate_sa.state})")  # Show the state for SA

def test_eight_queens():
    print("\n----- Eight Queens Local Search Test -----")
    board_size = 8
    num_restarts_queens = iterations_queens = 50
    q_problem_base = EightQueensProblem(size=board_size)
    print(f"Sample Initial {board_size}-Queens state: {q_problem_base.initial_state}")
    print(f"Sample Initial conflicts: {q_problem_base.evaluate(q_problem_base.initial_state)}")

    # --- Test Hill Climbing Variants ---
    candidate_steepest = local_search(EightQueensProblem(size=board_size), strategy="steepest")
    print(f"Steepest Ascent Final conflicts: {candidate_steepest.value} (Goal reached? {candidate_steepest.value == 0})")

    candidate_stochastic = local_search(EightQueensProblem(size=board_size), strategy="stochastic")
    print(f"Stochastic HC Final conflicts: {candidate_stochastic.value} (Goal reached? {candidate_stochastic.value == 0})")

    candidate_first_choice = local_search(EightQueensProblem(size=board_size), strategy="first_choice")
    print(f"First Choice HC Final conflicts: {candidate_first_choice.value} (Goal reached? {candidate_first_choice.value == 0})")

    # --- Test Random Restart ---
    candidate_restart_steepest = local_search(EightQueensProblem(size=board_size), strategy="random_restart", num_restarts=num_restarts_queens, base_restart_strategy="steepest")
    print(f"Random Restart (base: Steepest) Final conflicts: {candidate_restart_steepest.value} (Goal reached? {candidate_restart_steepest.value == 0})")

    candidate_restart_stochastic = local_search(EightQueensProblem(size=board_size), strategy="random_restart", num_restarts=num_restarts_queens, base_restart_strategy="stochastic")
    print(f"Random Restart (base: Stochastic) Final conflicts: {candidate_restart_stochastic.value} (Goal reached? {candidate_restart_stochastic.value == 0})")

    candidate_restart_first_choice = local_search(EightQueensProblem(size=board_size), strategy="random_restart", num_restarts=num_restarts_queens, base_restart_strategy="first_choice")
    print(f"Random Restart (base: First Choice) Final conflicts: {candidate_restart_first_choice.value} (Goal reached? {candidate_restart_first_choice.value == 0})")

    candidate_sa = local_search(EightQueensProblem(size=board_size), strategy="simulated_annealing")
    print(f"Simulated Annealing Final conflicts: {candidate_sa.value} (Goal reached? {candidate_sa.value == 0})")
    if candidate_sa.value == 0:
         print(f"(SA Solution state: {candidate_sa.state})")

# ------------------------------------------------------------------------------
# Main Execution Block
# ------------------------------------------------------------------------------
if __name__ == "__main__":
    # Optional: Seed random for reproducibility during testing
    

    test_tsp()
    test_eight_queens()



----- TSP Local Search Test -----
Sample Initial TSP tour: ['C', 'O', 'H', 'K', 'J', 'L', 'M', 'N', 'I', 'D', 'A', 'B', 'P', 'T', 'G', 'R', 'Q', 'S', 'E', 'F']
Sample Initial tour cost: 371.96

=== Running: Steepest Hill Climbing ===
Steepest Ascent Final tour cost: 340.02

=== Running: Stochastic Hill Climbing ===
Stochastic HC Final tour cost: 424.38

=== Running: First_choice Hill Climbing ===
First Choice HC Final tour cost: 333.44

=== Running: Random Restart Hill Climbing (50 restarts) ===
Random Restart (base: Steepest) Final tour cost: 252.32

=== Running: Random Restart Hill Climbing (50 restarts) ===
Random Restart (base: Stochastic) Final tour cost: 309.03

=== Running: Random Restart Hill Climbing (50 restarts) ===
Random Restart (base: First Choice) Final tour cost: 305.62

=== Running: Simulated Annealing ===
Simulated Annealing Final tour cost: 154.66
(Best SA state found: ['S', 'P', 'N', 'Q', 'O', 'M', 'K', 'I', 'F', 'D', 'B', 'A', 'C', 'E', 'G', 'H', 'J', 'L', 'R', 'T